# Quantum Subspace Expansion (QSE) on acrolein

This notebook demonstrates the Quantum Subspace Expansion (QSE) algorithm in `qibochem`.
We work with the acrolein molecule in the STO-3G basis set.

The workflow is:
1. Build the $H_2$ molecular Hamiltonian.
2. Prepare a reference state using the UCCSD ansatz with an MP2 initial guess of the parameters.
3. Run QSE using spin-conserving single-excitation operators as the expansion basis.
4. Compare the QSE energies against the exact eigenspectrum of the Hamiltonian.

In [1]:
from pathlib import Path

import numpy as np

from qibo.models import VQE

from qibochem.driver import Molecule
from qibochem.ansatz import ucc_ansatz
from qibochem.selected_ci.qse import QSE, QSEConfig

## 1. System Definition & Hamiltonian

We define the H₂ molecule at equilibrium bond length (0.735 Å) in the STO-3G basis set and run a PySCF calculation to obtain the molecular integrals.

In [2]:
# Define H2 at equilibrium bond length and run PySCF
mol = Molecule(
    geometry=[
        ["C", [-0.6377011674433, -0.4431524208018, 0.00000]],
        ["C", [0.5917425886738, 0.3634162575845, 0.00000]],
        ["C", [-1.8319770543167, 0.1529001654426, 0.00000]],
        ["O", [1.7078025982691, -0.1029927652550, 0.00000]],
        ["H", [0.4323601768503, 1.4573302862437, 0.00000]],
        ["H", [-0.5206284764655, -1.5184440344319, 0.00000]],
        ["H", [-1.9016075083300, 1.2335712530627, 0.00000]],
        ["H", [-2.7539328976638, -0.4090388222262, 0.00000]],
    ],
    basis="cc-pvdz"
)
mol.run_pyscf()

print(f"Number of spin-orbitals (qubits): {mol.nso}")
print(f"Number of electrons: {mol.nelec}")
print(f"Hartree-Fock energy: {mol.e_hf:.12f} Ha")

Number of spin-orbitals (qubits): 152
Number of electrons: 30
Hartree-Fock energy: -190.778458195622 Ha


In [3]:
# The MO coefficients from running a SA-CASSCF calculation are loaded
casscf_mo_file = Path("casscf_mos.npy")

if casscf_mo_file.is_file():
    casscf_mo = np.load(casscf_mo_file)
    # Update the MO coefficients and define the active space
    mol.ca = casscf_mo
else:
    print("casscf_mo_file not found")

mol.hf_embedding(active=[12, 14, 15])

## 2. Reference State: UCCSD Ansatz with MP2 Guess

We construct a UCCSD circuit using `qibochem`'s `ucc_ansatz` function. This:
- Starts from the Hartree-Fock reference state $|1100\rangle$
- Generates all single and double excitation operators under Jordan-Wigner mapping
- Initialises the circuit parameters using MP2 amplitudes (a physically motivated initial guess)

In [4]:
# Build UCCSD circuit seeded with MP2 amplitudes
circuit = ucc_ansatz(mol, excitations=[[2, 3, 4, 5], [0, 1, 4, 5]])

print(f"Number of qubits in circuit: {circuit.nqubits}")
print(f"Number of variational parameters: {len(circuit.get_parameters())}")
print(f"MP2 initial parameters: {circuit.get_parameters()}")

Number of qubits in circuit: 6
Number of variational parameters: 16
MP2 initial parameters: [((-0.0005420512641913049+0j),), ((0.0005420512641913049-0j),), ((0.0005420512641913049-0j),), ((0.0005420512641913049-0j),), ((-0.0005420512641913049+0j),), ((-0.0005420512641913049+0j),), ((-0.0005420512641913049+0j),), ((0.0005420512641913049-0j),), ((-8.024836615746774e-12+0j),), ((8.024836615746774e-12-0j),), ((8.024836615746774e-12-0j),), ((8.024836615746774e-12-0j),), ((-8.024836615746774e-12+0j),), ((-8.024836615746774e-12+0j),), ((-8.024836615746774e-12+0j),), ((8.024836615746774e-12-0j),)]


In [5]:
hamiltonian = mol.hamiltonian()
# Create VQE model
vqe = VQE(circuit, hamiltonian)

# Optimize starting from a random guess for the variational parameters
initial_parameters = np.random.uniform(0, 2 * np.pi, len(circuit.get_parameters()))
best, params, extra = vqe.minimize(initial_parameters, method='BFGS', compile=False)
circuit.set_parameters(params)

[Qibo 0.3.0|INFO|2026-03-18 11:36:39]: Using numpy backend on /CPU:0


## 3. Running QSE

The QSE algorithm expands the variational subspace by applying a set of excitation operators $\{E_a\}$ to the reference state $|\Psi\rangle$ and solving the generalised eigenvalue problem:

$$H_{ab} c = S_{ab} c \varepsilon$$

where $H_{ab} = \langle \Psi | E_a^\dagger H E_b | \Psi \rangle$ and $S_{ab} = \langle \Psi | E_a^\dagger E_b | \Psi \rangle$.

Here we use all spin-conserving one-body excitations ($E_a = a_i^\dagger a_j$ with $i,j$ same spin) as expansion operators.

Setting `n_shots=None` (the default) uses exact statevector simulation.

In [6]:
qse = QSE(mol, circuit)

n_shots = None
result = qse.run(circuit, n_shots)

print(f"QSE energies (Ha): {result.energies}")
print(f"Subspace dimension (before filtering): {result.subspace_dimension}")
print(f"Projected subspace dimension (after overlap filtering): {result.projected_subspace_dimension}")

QSE energies (Ha): [-190.76555268 -190.62612917 -190.44737606 -190.34524478 -190.31724811
 -189.88447269]
Subspace dimension (before filtering): 9
Projected subspace dimension (after overlap filtering): 6


In [7]:
# Print out the QSE results more nicely

TO_EV = 27.211386245981

def ordinal_label(n):
    """Print excited state energies in eV"""
    if n == 0:
        return "gs"
    suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    if 10 <= n % 100 <= 20:
        suffix = "th"
    return f"{n}{suffix}"

def qse_results(eigenvalues):
    """Print out the QSE results nicely"""
    print("Results wrt to QSE E0:")
    for i, energy in enumerate(eigenvalues):
        state_energy = (energy - eigenvalues[0]) * TO_EV
        print(f"QSE (shots) {ordinal_label(i+1)}: {state_energy:.4f} eV")

In [8]:
qse_results(result.energies)

Results wrt to QSE E0:
QSE (shots) 1st: 0.0000 eV
QSE (shots) 2nd: 3.7939 eV
QSE (shots) 3rd: 8.6580 eV
QSE (shots) 4th: 11.4372 eV
QSE (shots) 5th: 12.1990 eV
QSE (shots) 6th: 23.9754 eV


## 4. Comparison with Exact Eigenspectrum

Here we compute the exact energies corresponding to the qubit Hamiltonian for reference.

In [9]:
from openfermion.linalg import eigenspectrum

# Compute exact eigenvalues of the full qubit Hamiltonian
qubit_hamiltonian = mol.hamiltonian(ham_type="q", ferm_qubit_map="jw")
all_exact_energies = eigenspectrum(qubit_hamiltonian)

print("All exact eigenvalues (Ha):")
qse_results(all_exact_energies[:10])

All exact eigenvalues (Ha):
Results wrt to QSE E0:
QSE (shots) 1st: 0.0000 eV
QSE (shots) 2nd: 3.2578 eV
QSE (shots) 3rd: 3.2578 eV
QSE (shots) 4th: 3.2578 eV
QSE (shots) 5th: 3.2840 eV
QSE (shots) 6th: 3.2840 eV
QSE (shots) 7th: 3.7939 eV
QSE (shots) 8th: 6.9510 eV
QSE (shots) 9th: 6.9510 eV
QSE (shots) 10th: 6.9510 eV


Next we try running QSE with shots. There are 3 possible ways to do this:

1. Uniform shot allocation: Every group of Pauli terms is given `n_shots` shots for the expectation value calculation
2. Coefficient shot allocation: `n_shots` shots is distributed amongst the groups of Pauli terms based on the magnitude of their coefficients
3. Adaptive shot allocation: The shot distribution is further refined by considering the $S$ and $H$ matrices formed using the state vector corresponding to some guess circuit. Additionally, the code is currently set to iterate until the $S_1$ energy converges, up to a total of 10 tries.

In [10]:
# Uniform shot allocation
uniform = QSE(mol, circuit)

result = uniform.run(
    circuit,
    n_shots=10000  # Play around with this value
)

print(f"Total circuit runs: {result.total_circuit_runs}")
qse_results(result.energies)

Total circuit runs: 1690000
Results wrt to QSE E0:
QSE (shots) 1st: 0.0000 eV
QSE (shots) 2nd: 3.4570 eV
QSE (shots) 3rd: 5.0866 eV
QSE (shots) 4th: 9.9947 eV
QSE (shots) 5th: 13.7330 eV
QSE (shots) 6th: 21.5165 eV
QSE (shots) 7th: 27.1420 eV


In [11]:
# Coefficient shot allocation
allocated = QSE(mol, circuit)

result = allocated.run(
    circuit,
    n_shots=500000,  # Play around with this value
    uniform_shot_allocation=False,
    adaptive=False
)

print(f"Total circuit runs: {result.total_circuit_runs}")
qse_results(result.energies)

Total circuit runs: 500000
Results wrt to QSE E0:
QSE (shots) 1st: 0.0000 eV
QSE (shots) 2nd: 2.7244 eV
QSE (shots) 3rd: 6.5684 eV
QSE (shots) 4th: 10.8174 eV
QSE (shots) 5th: 16.2984 eV
QSE (shots) 6th: 27.8732 eV


In [12]:
# Uniform shot allocation
adaptive = QSE(mol, circuit)

result = adaptive.run(
    circuit,
    n_shots=50000,  # Play around with this value
    uniform_shot_allocation=False,
    adaptive=True
)

print(f"Total circuit runs: {result.total_circuit_runs}")
qse_results(result.energies)

Total circuit runs: 200000
Results wrt to QSE E0:
QSE (shots) 1st: 0.0000 eV
QSE (shots) 2nd: 0.4217 eV
QSE (shots) 3rd: 3.8395 eV
QSE (shots) 4th: 8.9770 eV
QSE (shots) 5th: 16.4412 eV
QSE (shots) 6th: 30.4237 eV
QSE (shots) 7th: 42.6306 eV
